# Threads and concurrency — waiting vs computing
Use the selected cells during session 2. Jupyter already has an asyncio event loop: use top-level `await`. Process workers live in the supplied importable module and use an explicit **spawn** context. A standalone script that starts a pool needs an `if __name__ == "__main__":` guard.

Real I/O has a labelled simulated fallback. CPU cells are independent: start at the worker setup if the network is unavailable.

In [ ]:
import urllib.request, zipfile, io, time, os
from datetime import date, timedelta
from concurrent.futures import ThreadPoolExecutor

# Set COURSE_LIVE_IO=0 before starting the kernel for the offline route.
LIVE_IO = os.environ.get("COURSE_LIVE_IO", "1") == "1"
days = [date(2024, 6, 1) + timedelta(d) for d in range(20)]

def close_of(day):
    if not LIVE_IO:
        time.sleep(0.05)  # controlled waiting, NOT a network benchmark
        return 100.0 + (day - days[0]).days  # synthetic values, NOT Bitcoin prices
    url = f'https://data.binance.vision/data/spot/daily/klines/BTCUSDT/1d/BTCUSDT-1d-{day}.zip'
    with urllib.request.urlopen(url, timeout=3) as response:
        raw = response.read()
    with zipfile.ZipFile(io.BytesIO(raw)) as archive:
        row = archive.read(archive.namelist()[0]).decode().splitlines()[0]
    return float(row.split(',')[4])

def timed(fn):
    start = time.perf_counter()
    result = fn()
    return result, time.perf_counter() - start

# One probe: on failure use the controlled-wait exercise for the whole comparison.
if LIVE_IO:
    try:
        close_of(days[0])
    except (OSError, ValueError, zipfile.BadZipFile) as error:
        print("Archive unavailable:", type(error).__name__)
        LIVE_IO = False
print("REAL archived prices" if LIVE_IO else "SIMULATED waits + synthetic values")


In [ ]:
import asyncio

async def main():
    results = await asyncio.gather(*(asyncio.to_thread(close_of, day) for day in days), return_exceptions=True)
    for result in results:
        if isinstance(result, Exception):
            raise result
    return results

# Wrong timer: it stops at submission. Still join tasks before the next comparison.
async def bad_bench():
    start = time.perf_counter()
    tasks = [asyncio.create_task(asyncio.to_thread(close_of, day)) for day in days]
    submitted = time.perf_counter() - start
    await asyncio.gather(*tasks, return_exceptions=True)
    return submitted

try:
    sequential, t_seq = timed(lambda: [close_of(day) for day in days])
    t_bad = await bad_bench()
    start = time.perf_counter()
    good = await main()
    t_async = time.perf_counter() - start
except (OSError, ValueError, zipfile.BadZipFile):
    LIVE_IO = False
    print("Restarting BOTH timings: SIMULATED waits + synthetic values")
    sequential, t_seq = timed(lambda: [close_of(day) for day in days])
    t_bad = await bad_bench()
    start = time.perf_counter()
    good = await main()
    t_async = time.perf_counter() - start
    assert good == sequential
    print("simulation", "seq", t_seq, "submission only", t_bad, "awaited", t_async)
else:
    assert good == sequential
    print("real archive" if LIVE_IO else "simulation", "seq", t_seq, "submission only", t_bad, "awaited", t_async)


## Pure Python CPU loops
The GIL serializes Python bytecode in ordinary CPython builds. It does **not** mean that all NumPy or other native work is serialized. Predict threads vs processes; all timings below include executor creation and shutdown.

In [ ]:
# Provided setup: workers must be importable by a fresh Python process.
from pathlib import Path
import sys, urllib.request
candidates = [Path.cwd(), *list(Path.cwd().parents)[:4]]
worker_dir = next((p / suffix for p in candidates
                   for suffix in ("", "python", "teaching/parallel_computing/python")
                   if (p / suffix / "parallel_workers.py").is_file()), None)
if worker_dir is None:  # a standalone notebook, e.g. Colab
    url = "https://raw.githubusercontent.com/epilliat/epilliat.github.io/master/docs/teaching/parallel_computing/python/parallel_workers.py"
    urllib.request.urlretrieve(url, "parallel_workers.py")
    worker_dir = Path.cwd()
sys.path.insert(0, str(worker_dir))
from parallel_workers import cpu_load, count_in_disk, make_jobs
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
import multiprocessing as mp
import time, platform
print(platform.python_version(), "start method: spawn; timings include pool startup")


In [ ]:
iterations = [500_000] * 8
reference, sequential_time = timed(lambda: list(map(cpu_load, iterations)))
for executor in (ThreadPoolExecutor, ProcessPoolExecutor):
    start = time.perf_counter()
    options = {"mp_context": mp.get_context("spawn")} if executor is ProcessPoolExecutor else {}
    with executor(max_workers=4, **options) as pool:
        values = list(pool.map(cpu_load, iterations))
    elapsed = time.perf_counter() - start
    assert values == reference
    print(executor.__name__, elapsed, "speedup", sequential_time/elapsed)


## Monte-Carlo π: independent counts, one reduction
Use fixed lots and seeds when varying the worker count; include the remainder. Reproducibility refers to this partition and environment, not identical results across all versions/machines. Independently seeded generators are intended to avoid correlated streams with very high probability; this is not a mathematical proof of independence.

In [ ]:
jobs = make_jobs(1_000_003, lots=8)
reference = list(map(count_in_disk, jobs))
with ProcessPoolExecutor(max_workers=4, mp_context=mp.get_context("spawn")) as pool:
    counts = list(pool.map(count_in_disk, jobs))
assert counts == reference
print("π ≈", 4 * sum(counts) / sum(n for n, seed in jobs))


## Summary
Waiting: asyncio or threads can overlap I/O. Pure Python CPU loops under the GIL: processes can help **if work amortizes overhead**. Native NumPy CPU work: threads may also help; control BLAS threads to avoid oversubscription. Julia tasks return local counts and reduce once, with `Threads.@spawn` for CPU work. See the separate CPU activity; a race needs a shared mutable counter, not simply multiple tasks.